# QASPER Model Evaluation

Public portfolio edition prepared for GitHub and Databricks. Credentials are read from environment variables; research data and generated artifacts are not committed to Git.


In [ ]:
# Databricks uses Unity Catalog Volumes; no Google Drive mount is required.

import os
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "")

from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN"))

In [ ]:
# 安装依赖（如当前环境尚未安装）
!pip -q install ragas "datasets>=2.19" langchain-community langchain-openai tiktoken rouge-score

import os, re, numpy as np, pandas as pd, tiktoken
from pathlib import Path
from datasets import Dataset
from rouge_score import rouge_scorer
from langchain_community.embeddings import HuggingFaceEmbeddings

# 模型轮换池（按你的限额友好顺序）
os.environ["RAGAS_MODEL_POOL"] = "gpt-4.1-nano,gpt-5-nano,gpt-4o-mini,gpt-4.1-mini,gpt-3.5-turbo"

INP = Path("/Volumes/main/default/thesis_project/Evaluation/Eval_Test/qasper_all_models_unified.1.7.parquet")
DF  = pd.read_parquet(INP).copy()

# has_retrieval
DF["has_retrieval"] = DF["retrieved_ctx"].apply(lambda x: isinstance(x,(list,tuple)) and len(x)>0)

# ---- 构建 top-k（空时回退第一段，避免 ragas 拿到空）----
ENC = tiktoken.get_encoding("cl100k_base")
def _tok_len(s): return len(ENC.encode(s or ""))
def _truncate(s, max_toks):
    ids = ENC.encode(s or "");
    return s if len(ids)<=max_toks else ENC.decode(ids[:max_toks])

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
def _select_topk(ev_list, q, a, k=3, budget=600, per_ctx=300):
    if not isinstance(ev_list,(list,tuple)): return []
    evs = [e for e in ev_list if isinstance(e,str) and e.strip()]
    if not evs: return []
    scored=[]
    for ev in evs:
        s = 0.6*scorer.score(q,ev)["rougeL"].fmeasure + 0.4*scorer.score(a,ev)["rougeL"].fmeasure
        scored.append((s,ev))
    scored.sort(key=lambda x:x[0], reverse=True)
    out, tot = [], 0
    for _,ev in scored[:k*3]:
        ev2 = _truncate(ev, per_ctx)
        t = _tok_len(ev2)
        if tot+t <= budget:
            out.append(ev2); tot+=t
        if len(out)>=k: break
    if not out:  # 回退至少保留第一段
        out = [_truncate(evs[0], min(300, budget))]
    return out[:k]

DF["_ref_topk"] = DF.apply(lambda r: _select_topk(r.get("oracle_ctx",[]),    r["question"], r["answer"]), axis=1)
DF["_ctx_topk"] = DF.apply(lambda r: _select_topk(r.get("retrieved_ctx",[]), r["question"], r["answer"]), axis=1)

# 仅挑“retrieved/ref 同时非空”的 10 条做探针
mask_both = DF["_ctx_topk"].apply(lambda x: isinstance(x,(list,tuple)) and len(x)>0) & \
            DF["_ref_topk"].apply(lambda x: isinstance(x,(list,tuple)) and len(x)>0)
TEST = DF[mask_both].sample(min(10, mask_both.sum()), random_state=42).copy()

# —— ragas 期望字段名
def _join_ref(lst):
    if isinstance(lst,(list,tuple)):
        return "\n\n".join(str(x) for x in lst if str(x).strip())
    return str(lst or "")

from ragas.metrics import answer_relevancy as M_ANS_REL
from ragas.metrics import faithfulness    as M_FAITH
M_CP = M_CR = M_CREL = None
try:
    from ragas.metrics import context_precision as M_CP
except Exception: pass
try:
    from ragas.metrics import context_recall as M_CR
except Exception: pass
try:
    from ragas.metrics import context_relevancy as M_CREL
except Exception: pass

from ragas import evaluate
try:
    from langchain_openai import ChatOpenAI
    model0 = os.getenv("RAGAS_MODEL_POOL").split(",")[0].strip()
    llm = ChatOpenAI(model=model0, temperature=0)
except Exception:
    from langchain.chat_models import ChatOpenAI
    llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

emb = HuggingFaceEmbeddings(model_name="intfloat/e5-base-v2")

# Retrieval 探针
probe_ds_retr = Dataset.from_dict({
    "user_input":         TEST["question"].astype(str).tolist(),
    "response":           TEST["answer"].astype(str).tolist(),
    "retrieved_contexts": TEST["_ctx_topk"].tolist(),
    "reference":          [ _join_ref(x) for x in TEST["_ref_topk"].tolist() ],
})
metrics_r = [M_ANS_REL, M_FAITH] + [m for m in [M_CREL,M_CP,M_CR] if m is not None]
res_r = evaluate(probe_ds_retr, metrics=metrics_r, llm=llm, embeddings=emb).to_pandas()
print("Retrieval probe columns:", list(res_r.columns))
display(pd.concat([TEST[["model","question_id","question","answer"]].reset_index(drop=True),
                   res_r[[c for c in res_r.columns if c in ["answer_relevancy","faithfulness","context_relevancy","context_precision","context_recall"]]]], axis=1))


In [ ]:
# --- 修复探针：用 M1 的 oracle_evidence 回填 oracle_ctx，保证抽样非空；必要时降级指标 ---
import os, re, numpy as np, pandas as pd, tiktoken
from pathlib import Path
from datasets import Dataset
from rouge_score import rouge_scorer
from langchain_community.embeddings import HuggingFaceEmbeddings

UNIFIED = Path("/Volumes/main/default/thesis_project/Evaluation/Eval_Test/qasper_all_models_unified.1.7.parquet")
M1_PATH = Path("/Volumes/main/default/thesis_project/M1/Test_2.2/qasper_test_M1_answers_sample_2.2.parquet")

DF = pd.read_parquet(UNIFIED).copy()

# 1) 用 M1 的 oracle_evidence 回填 oracle_ctx（只对缺失或空列表的行）
def _norm_q(s):
    return re.sub(r"\s+"," ", str(s or "").strip().lower())

DF["__qkey__"] = DF["question"].apply(_norm_q)

if M1_PATH.exists():
    M1 = pd.read_parquet(M1_PATH)[["question","oracle_evidence"]].copy()
    M1["__qkey__"] = M1["question"].apply(_norm_q)
    # 每个问题取第一条非空 evidence
    m1_map = (M1.dropna(subset=["oracle_evidence"])
                .groupby("__qkey__", sort=False)["oracle_evidence"]
                .apply(lambda s: next((v for v in s if isinstance(v,str) and v.strip()), np.nan)))
    # 回填
    if "oracle_ctx" not in DF.columns:
        DF["oracle_ctx"] = [[] for _ in range(len(DF))]
    def _fill_oracle_ctx(row):
        cur = row["oracle_ctx"]
        if isinstance(cur, (list, tuple)) and len(cur)>0:
            return list(cur)
        ev = m1_map.get(row["__qkey__"], np.nan)
        return [ev] if isinstance(ev, str) and ev.strip() else []
    DF["oracle_ctx"] = DF.apply(_fill_oracle_ctx, axis=1)
else:
    print("⚠️ 未找到 M1 文件，跳过 oracle_evidence 回填。")

# 2) has_retrieval 标注
DF["has_retrieval"] = DF["retrieved_ctx"].apply(lambda x: isinstance(x,(list,tuple)) and len(x)>0)

# 3) 构建 top-k（空时回退第一段，避免 ragas 见空）
ENC = tiktoken.get_encoding("cl100k_base")
def _tok_len(s): return len(ENC.encode(s or ""))
def _truncate(s, max_toks):
    ids = ENC.encode(s or "");
    return s if len(ids)<=max_toks else ENC.decode(ids[:max_toks])

_scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
def _topk(ev_list, q, a, k=3, budget=600, per_ctx=300):
    if not isinstance(ev_list,(list,tuple)): return []
    evs = [e for e in ev_list if isinstance(e,str) and e.strip()]
    if not evs: return []
    scored = []
    for ev in evs:
        s = 0.6*_scorer.score(q,ev)["rougeL"].fmeasure + 0.4*_scorer.score(a,ev)["rougeL"].fmeasure
        scored.append((s,ev))
    scored.sort(key=lambda x:x[0], reverse=True)
    out, tot = [], 0
    for _,ev in scored[:k*3]:
        ev2 = _truncate(ev, per_ctx)
        t = _tok_len(ev2)
        if tot+t <= budget:
            out.append(ev2); tot += t
        if len(out)>=k: break
    if not out:
        out = [_truncate(evs[0], min(300, budget))]
    return out[:k]

DF["_ref_topk"] = DF.apply(lambda r: _topk(r.get("oracle_ctx",[]),    r["question"], r["answer"]), axis=1)
DF["_ctx_topk"] = DF.apply(lambda r: _topk(r.get("retrieved_ctx",[]), r["question"], r["answer"]), axis=1)

# 4) 统计可用行
has_ref   = DF["_ref_topk"].apply(lambda x: isinstance(x,(list,tuple)) and len(x)>0)
has_ctx   = DF["_ctx_topk"].apply(lambda x: isinstance(x,(list,tuple)) and len(x)>0)
both_mask = has_ref & has_ctx

print(f"可用参考上下文(ref) 行数：{int(has_ref.sum())}")
print(f"可用检索上下文(ctx) 行数：{int(has_ctx.sum())}")
print(f"两者同时可用(both) 行数：{int(both_mask.sum())}")

# 5) 构造探针数据集（优先 both，否则仅 ctx）
if both_mask.any():
    TEST = DF[both_mask].sample(min(10, int(both_mask.sum())), random_state=42).copy()
    mode = "full"   # 跑 answer_relevancy/faithfulness/context_precision/recall
else:
    # 退化为只评 answer_relevancy + faithfulness（不需要 reference）
    ctx_only = has_ctx
    if not ctx_only.any():
        raise RuntimeError("没有任何行具有检索上下文，无法做 retrieval 探针。请检查 retrieved_ctx 是否已写入。")
    TEST = DF[ctx_only].sample(min(10, int(ctx_only.sum())), random_state=42).copy()
    mode = "ctx_only"

# 6) ragas 评测（探针 10 条）
from ragas.metrics import answer_relevancy as M_ANS_REL
from ragas.metrics import faithfulness    as M_FAITH
M_CP = M_CR = M_CREL = None
try:
    from ragas.metrics import context_precision as M_CP
except Exception: pass
try:
    from ragas.metrics import context_recall as M_CR
except Exception: pass
try:
    from ragas.metrics import context_relevancy as M_CREL
except Exception: pass

def _join_ref(lst):
    if isinstance(lst,(list,tuple)):
        return "\n\n".join(str(x) for x in lst if str(x).strip())
    return str(lst or "")

try:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model=os.getenv("RAGAS_MODEL_POOL","gpt-4.1-nano,gpt-5-nano,gpt-4o-mini,gpt-4.1-mini,gpt-3.5-turbo").split(",")[0], temperature=0)
except Exception:
    from langchain.chat_models import ChatOpenAI
    llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

emb = HuggingFaceEmbeddings(model_name="intfloat/e5-base-v2")

from ragas import evaluate

if mode == "full":
    probe = Dataset.from_dict({
        "user_input":         TEST["question"].astype(str).tolist(),
        "response":           TEST["answer"].astype(str).tolist(),
        "retrieved_contexts": TEST["_ctx_topk"].tolist(),
        "reference":          [ _join_ref(x) for x in TEST["_ref_topk"].tolist() ],
    })
    metrics = [M_ANS_REL, M_FAITH] + [m for m in [M_CREL,M_CP,M_CR] if m is not None]
else:
    probe = Dataset.from_dict({
        "user_input":         TEST["question"].astype(str).tolist(),
        "response":           TEST["answer"].astype(str).tolist(),
        "retrieved_contexts": TEST["_ctx_topk"].tolist(),
    })
    metrics = [M_ANS_REL, M_FAITH]   # 不包含需要 reference 的指标

print(f"探针模式: {mode}  | 样本数: {len(TEST)}")
res = evaluate(probe, metrics=metrics, llm=llm, embeddings=emb).to_pandas()
print("得到列：", list(res.columns))

keep = [c for c in ["answer_relevancy","faithfulness","context_relevancy","context_precision","context_recall"] if c in res.columns]
display(pd.concat([TEST[["model","question_id","question","answer"]].reset_index(drop=True),
                   res[keep]], axis=1))


In [ ]:
import pandas as pd, numpy as np, json, ast, re
from pathlib import Path

INP  = Path("/Volumes/main/default/thesis_project/Evaluation/Eval_Test/qasper_all_models_unified.1.7.parquet")
OUTP = Path("/Volumes/main/default/thesis_project/Evaluation/Eval_Test/qasper_all_models_unified.1.7.fix.parquet")

DF = pd.read_parquet(INP)
print("rows:", len(DF), "| has retrieved_ctx:", "retrieved_ctx" in DF.columns)

# 诊断：看看前几项的真实类型/内容
def _peek(v, n=120):
    s = repr(v)
    return s if len(s) <= n else s[:n] + "…"
print("\n[peek retrieved_ctx head]:")
for i, v in enumerate(DF.get("retrieved_ctx", [])[:5]):
    print(f"  [{i}] type={type(v).__name__} | {_peek(v)}")

# 解析器：把字符串/JSON/分隔串统一成 list[str]
def _coerce_list_any(x):
    if isinstance(x, list):
        return [str(t) for t in x if str(t).strip()]
    if isinstance(x, tuple):
        return [str(t) for t in x if str(t).strip()]
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    s = str(x).strip()
    # JSON 列表或对象
    if (s.startswith("[") and s.endswith("]")) or (s.startswith("{") and s.endswith("}")):
        try:
            obj = json.loads(s)
            if isinstance(obj, list):
                return [str(t) for t in obj if str(t).strip()]
            if isinstance(obj, dict):
                # 兼容可能存了 {"contexts":[...]}
                if "contexts" in obj and isinstance(obj["contexts"], list):
                    return [str(t) for t in obj["contexts"] if str(t).strip()]
        except Exception:
            # Python 风格列表字符串
            try:
                obj = ast.literal_eval(s)
                if isinstance(obj, (list, tuple)):
                    return [str(t) for t in obj if str(t).strip()]
            except Exception:
                pass
    # 常见分隔符
    if "|||" in s:
        return [p.strip() for p in s.split("|||") if p.strip()]
    if "\n" in s:
        return [p.strip() for p in s.split("\n") if p.strip()]
    if "\u2029" in s:  # 段落分隔符
        return [p.strip() for p in s.split("\u2029") if p.strip()]
    # 兜底：空
    return []

if "retrieved_ctx" not in DF.columns:
    DF["retrieved_ctx"] = [[] for _ in range(len(DF))]
else:
    DF["retrieved_ctx"] = DF["retrieved_ctx"].apply(_coerce_list_any)

# 统计非空
nonempty = int(DF["retrieved_ctx"].apply(lambda x: isinstance(x, list) and len(x)>0).sum())
print("\nnon-empty retrieved_ctx rows:", nonempty)

# 每个模型下非空条数
if "model" in DF.columns:
    tmp = DF.assign(nonempty=DF["retrieved_ctx"].apply(lambda x: int(isinstance(x, list) and len(x)>0)))
    print("\nby model non-empty:")
    print(tmp.groupby("model")["nonempty"].sum())

# 回写
OUTP.parent.mkdir(parents=True, exist_ok=True)
DF.to_parquet(OUTP, index=False)
print("\n💾 wrote:", OUTP)


In [ ]:
# 如未安装请先装；已装会跳过
!pip -q install ragas "datasets>=2.19" langchain-community langchain-openai tiktoken rouge-score

import os, re, numpy as np, pandas as pd, tiktoken
from pathlib import Path
from datasets import Dataset
from rouge_score import rouge_scorer
from langchain_community.embeddings import HuggingFaceEmbeddings

os.environ["RAGAS_MODEL_POOL"] = "gpt-4.1-nano,gpt-5-nano,gpt-4o-mini,gpt-4.1-mini,gpt-3.5-turbo"

INP = Path("/Volumes/main/default/thesis_project/Evaluation/Eval_Test/qasper_all_models_unified.1.7.fix.parquet")
DF  = pd.read_parquet(INP).copy()

# —— 保证 oracle_ctx/retrieved_ctx 是 list[str]
def _as_list(x):
    if isinstance(x, (list, tuple)): return [str(t) for t in x if str(t).strip()]
    # 兼容 numpy.ndarray
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray): return [str(t) for t in x.tolist() if str(t).strip()]
    except Exception:
        pass
    return []
DF["oracle_ctx"]    = DF.get("oracle_ctx", [[]]*len(DF)).apply(_as_list)
DF["retrieved_ctx"] = DF.get("retrieved_ctx", [[]]*len(DF)).apply(_as_list)

# —— 构建 top-k（ROUGE 选段 + token 预算）
ENC = tiktoken.get_encoding("cl100k_base")
def _tok_len(s): return len(ENC.encode(s or ""))
def _truncate(s, max_toks):
    ids = ENC.encode(s or "");
    return s if len(ids)<=max_toks else ENC.decode(ids[:max_toks])

_scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
def _topk(evs, q, a, k=3, budget=600, per_ctx=300):
    evs = [e for e in (evs or []) if isinstance(e,str) and e.strip()]
    if not evs: return []
    scored=[]
    for ev in evs:
        s = 0.6*_scorer.score(q,ev)["rougeL"].fmeasure + 0.4*_scorer.score(a,ev)["rougeL"].fmeasure
        scored.append((s,ev))
    scored.sort(key=lambda x:x[0], reverse=True)
    out, tot = [], 0
    for _,ev in scored[:k*3]:
        ev2=_truncate(ev, per_ctx); t=_tok_len(ev2)
        if tot+t<=budget: out.append(ev2); tot+=t
        if len(out)>=k: break
    if not out: out=[_truncate(evs[0], min(300,budget))]
    return out[:k]

DF["_ref_topk"] = DF.apply(lambda r: _topk(r["oracle_ctx"],    r["question"], r["answer"]), axis=1)
DF["_ctx_topk"] = DF.apply(lambda r: _topk(r["retrieved_ctx"], r["question"], r["answer"]), axis=1)

has_ref = DF["_ref_topk"].apply(lambda x: len(x)>0)
has_ctx = DF["_ctx_topk"].apply(lambda x: len(x)>0)
both    = has_ref & has_ctx
print(f"ref non-empty: {int(has_ref.sum())} | ctx non-empty: {int(has_ctx.sum())} | both: {int(both.sum())}")

# —— 采样 10 条（优先 both）
TEST = DF[both].sample(min(10, int(both.sum())), random_state=42).copy()
if len(TEST)==0:
    raise RuntimeError("没有同时具备 ctx/ref 的样本，请检查 oracle_ctx 是否已填；（M1 的 oracle_evidence 可回填）")

# —— ragas 指标
from ragas.metrics import answer_relevancy as M_ANS_REL
from ragas.metrics import faithfulness    as M_FAITH
M_CP = M_CR = M_CREL = None
try:
    from ragas.metrics import context_precision as M_CP
except Exception: pass
try:
    from ragas.metrics import context_recall as M_CR
except Exception: pass
try:
    from ragas.metrics import context_relevancy as M_CREL
except Exception: pass

def _join_ref(lst): return "\n\n".join(lst)

from ragas import evaluate
try:
    from langchain_openai import ChatOpenAI
    model0 = os.getenv("RAGAS_MODEL_POOL").split(",")[0].strip()
    llm = ChatOpenAI(model=model0, temperature=0)
except Exception:
    from langchain.chat_models import ChatOpenAI
    llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

emb = HuggingFaceEmbeddings(model_name="intfloat/e5-base-v2")

probe = Dataset.from_dict({
    "user_input":         TEST["question"].astype(str).tolist(),
    "response":           TEST["answer"].astype(str).tolist(),
    "retrieved_contexts": TEST["_ctx_topk"].tolist(),
    "reference":          [ _join_ref(x) for x in TEST["_ref_topk"].tolist() ],
})
metrics = [M_ANS_REL, M_FAITH] + [m for m in [M_CREL,M_CP,M_CR] if m is not None]
res = evaluate(probe, metrics=metrics, llm=llm, embeddings=emb).to_pandas()

keep = [c for c in ["answer_relevancy","faithfulness","context_relevancy","context_precision","context_recall"] if c in res.columns]
print("got cols:", keep)
display(pd.concat([TEST[["model","question_id","question","answer"]].reset_index(drop=True), res[keep]], axis=1))


In [ ]:
# 复用你刚才已加载好的 DF（来自 1.7.fix），且已构建好 _ctx_topk / _ref_topk
# 如果是在全新会话，请先重新读取 1.7.fix 并按探针单元格构建 _ctx_topk / _ref_topk

import os, math, numpy as np, pandas as pd
from pathlib import Path
from datasets import Dataset
from ragas import evaluate
from langchain_community.embeddings import HuggingFaceEmbeddings

# 模型池（从省配额到更强）
os.environ["RAGAS_MODEL_POOL"] = "gpt-4.1-nano,gpt-5-nano,gpt-4o-mini,gpt-4.1-mini,gpt-3.5-turbo"
MODEL_POOL = [m.strip() for m in os.getenv("RAGAS_MODEL_POOL").split(",") if m.strip()]

# 指标
from ragas.metrics import answer_relevancy as M_ANS_REL
from ragas.metrics import faithfulness    as M_FAITH
M_CP = M_CR = M_CREL = None
try:
    from ragas.metrics import context_precision as M_CP
except Exception:
    pass
try:
    from ragas.metrics import context_recall as M_CR
except Exception:
    pass
try:
    from ragas.metrics import context_relevancy as M_CREL
except Exception:
    pass

def _join_ref(lst):
    return "\n\n".join([x for x in (lst or []) if isinstance(x,str) and x.strip()])

# 嵌入
emb = HuggingFaceEmbeddings(model_name="intfloat/e5-base-v2")

# LLM 构造
def _make_llm(model_name):
    try:
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=model_name, temperature=0)
    except Exception:
        from langchain.chat_models import ChatOpenAI
        return ChatOpenAI(model_name=model_name, temperature=0)

def _eval_block(df_block, model_name: str):
    data = {
        "user_input":         df_block["question"].astype(str).tolist(),
        "response":           df_block["answer"].astype(str).tolist(),
        "retrieved_contexts": df_block["_ctx_topk"].tolist(),
        "reference":          [ _join_ref(x) for x in df_block["_ref_topk"].tolist() ],
    }
    ds = Dataset.from_dict(data)
    llm = _make_llm(model_name)
    metrics = [M_ANS_REL, M_FAITH] + [m for m in [M_CREL, M_CP, M_CR] if m is not None]
    return evaluate(ds, metrics=metrics, llm=llm, embeddings=emb).to_pandas()

def _eval_with_pool(df_block):
    last_err = None
    for m in MODEL_POOL:
        try:
            pdf = _eval_block(df_block, m)
            print(f"✅ retrieval block with {m}")
            return pdf
        except Exception as e:
            last_err = e
            s = str(e).lower()
            if ("rate limit" in s) or ("429" in s) or ("timeout" in s):
                print(f"↪️ rate/timeout on {m}; trying next…")
                continue
            else:
                print(f"❗ non-retryable on {m}: {e}")
                break
    raise RuntimeError(f"All models failed (retrieval). Last error: {last_err}")

def run_ragas_retrieval_only(df: pd.DataFrame, chunk_size: int = 150) -> pd.DataFrame:
    out = df.copy()
    mask = df["_ctx_topk"].apply(lambda x: isinstance(x,(list,tuple)) and len(x)>0)
    sub = df[mask]
    parts = []
    for i in range(0, len(sub), chunk_size):
        blk = sub.iloc[i:i+chunk_size]
        pdf = _eval_with_pool(blk)
        pdf.index = blk.index
        parts.append(pdf)
    R = pd.concat(parts).sort_index()
    # 写回前缀列
    if "answer_relevancy" in R:  out.loc[mask, "retrieval::answer_relevancy"] = R["answer_relevancy"]
    if "faithfulness"     in R:  out.loc[mask, "retrieval::faithfulness"]     = R["faithfulness"]
    if "context_relevancy" in R: out.loc[mask, "retrieval::context_relevancy"] = R["context_relevancy"]
    if "context_precision" in R: out.loc[mask, "retrieval::context_precision"] = R["context_precision"]
    if "context_recall"    in R: out.loc[mask, "retrieval::context_recall"]    = R["context_recall"]
    return out

# ==== 跑起来（仅 retrieval） ====
DF_retr = run_ragas_retrieval_only(DF, chunk_size=150)

# 保存
OUT = Path("/Volumes/main/default/thesis_project/Evaluation/Eval_Test/qasper_all_models_unified.scored.retrieval.1.5.parquet")
DF_retr.to_parquet(OUT, index=False)
print("💾 Saved:", OUT)

# 快速核验
cols = ["retrieval::answer_relevancy","retrieval::faithfulness",
        "retrieval::context_relevancy","retrieval::context_precision","retrieval::context_recall"]
exist = [c for c in cols if c in DF_retr.columns]
print("Non-null counts:")
print(DF_retr[exist].notna().sum())
if "model" in DF_retr.columns:
    print("\nPer-model means (retrieval rows only):")
    mask = DF_retr["_ctx_topk"].apply(lambda x: isinstance(x,(list,tuple)) and len(x)>0)
    display(DF_retr[mask].groupby("model")[exist].mean(numeric_only=True).round(4))

In [ ]:
import ragas, importlib
print("ragas version:", ragas.__version__)
try:
    M_CREL = importlib.import_module("ragas.metrics").context_relevancy
    print("context_relevancy ✅ 可用")
except Exception as e:
    print("context_relevancy ❌ 不可用：", e)
    print("提示：如需该指标，可尝试： !pip install -U ragas")

In [ ]:
# === 只补 retrieval::context_relevancy ===
# MODE: "e5"（默认，省钱）或 "llm"（更接近原 ragas 思路，少量token）
MODE = "e5"   # 改成 "llm" 可切换为 LLM 版本

INP  = "/Volumes/main/default/thesis_project/Evaluation/Eval_Test/qasper_all_models_unified.scored.retrieval.1.5.parquet"
OUT  = "/Volumes/main/default/thesis_project/Evaluation/Eval_Test/qasper_all_models_unified.scored.retrieval.1.5.ctxrel.parquet"

import os, re, json, ast, math, time, numpy as np, pandas as pd

DF = pd.read_parquet(INP).copy()

# ------- 保证 _ctx_topk 是 list[str] -------
def _as_list(x):
    if isinstance(x, list): return [str(t) for t in x if str(t).strip()]
    if isinstance(x, tuple): return [str(t) for t in x if str(t).strip()]
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray): return [str(t) for t in x.tolist() if str(t).strip()]
    except Exception: pass
    if x is None: return []
    s = str(x).strip()
    if s.startswith("[") and s.endswith("]"):
        try:
            obj = json.loads(s)
            if isinstance(obj, list): return [str(t) for t in obj if str(t).strip()]
        except Exception:
            try:
                obj = ast.literal_eval(s)
                if isinstance(obj, (list, tuple)): return [str(t) for t in obj if str(t).strip()]
            except Exception: pass
    return []

if "_ctx_topk" not in DF.columns:
    # 如果没有 _ctx_topk，就从 retrieved_ctx 直接使用（不截断）
    DF["_ctx_topk"] = DF.get("retrieved_ctx", [[]]*len(DF)).apply(_as_list)
else:
    DF["_ctx_topk"] = DF["_ctx_topk"].apply(_as_list)

mask = DF["_ctx_topk"].apply(lambda x: isinstance(x, (list,tuple)) and len(x)>0)

print(f"Rows to compute ctx_relevancy: {int(mask.sum())} / {len(DF)}")

In [ ]:
if MODE == "e5":
    !pip -q install sentence-transformers
    import torch
    from sentence_transformers import SentenceTransformer
    from numpy.linalg import norm

    model = SentenceTransformer("intfloat/e5-base-v2", device="cuda" if torch.cuda.is_available() else "cpu")

    def _cos(a, b):
        an = norm(a); bn = norm(b)
        if an==0 or bn==0: return np.nan
        return float(np.dot(a, b) / (an*bn))

    # 阈值：0.35 是 E5 问答相关性的常见起点；你也可以改为 0.30~0.40 调整灵敏度
    THRESH = 0.35

    # --- 编码所有问题（query 前缀）---
    q_texts = DF["question"].astype(str).tolist()
    QV = model.encode([f"query: {q}" for q in q_texts], normalize_embeddings=True, show_progress_bar=True)

    # --- 展平所有上下文并编码（passage 前缀）---
    row_ids, passages = [], []
    for idx, ctxs in DF["_ctx_topk"].items():
        for p in (ctxs or []):
            passages.append(p)
            row_ids.append(idx)

    PV = model.encode([f"passage: {p}" for p in passages], normalize_embeddings=True, show_progress_bar=True)

    # --- 聚合为“相关段占比” ---
    hits = {}
    totals = {}
    for i, rid in enumerate(row_ids):
        s = _cos(QV[rid], PV[i])
        totals[rid] = totals.get(rid, 0) + 1
        if s >= THRESH:
            hits[rid] = hits.get(rid, 0) + 1

    ctxrel = np.full(len(DF), np.nan, dtype=float)
    for rid, tot in totals.items():
        ctxrel[rid] = (hits.get(rid, 0) / tot) if tot > 0 else np.nan

    DF["retrieval::context_relevancy"] = ctxrel

In [ ]:
# ------- 保存并小结 -------
DF.to_parquet(OUT, index=False)
print("💾 Saved:", OUT)

if "model" in DF.columns:
    sub = DF[DF["_ctx_topk"].apply(lambda x: isinstance(x, (list,tuple)) and len(x)>0)]
    print("\nPer-model means (retrieval rows):")
    print(sub.groupby("model")[["retrieval::context_relevancy"]].mean(numeric_only=True).round(4))
print("\nNon-null count:", DF["retrieval::context_relevancy"].notna().sum())

In [ ]:
# ============================
# Oracle RAGAS + E5 + AIS
# 继续在已完成的 retrieval 结果上计算
# ============================

!pip -q install ragas "datasets>=2.19" langchain-community langchain-openai tiktoken \
                 rouge-score sentence-transformers transformers torch --upgrade

import os, re, math, time, json, ast, numpy as np, pandas as pd, torch
from pathlib import Path
from tqdm.auto import tqdm

INP  = Path("/Volumes/main/default/thesis_project/Evaluation/Eval_Test/qasper_all_models_unified.scored.retrieval.1.5.parquet")
OUT  = Path("/Volumes/main/default/thesis_project/Evaluation/Eval_Test/qasper_all_models_unified.scored.full.1.16.parquet")
TMP  = Path("/Volumes/main/default/thesis_project/Evaluation/Eval_Test/_tmp_oracle_eval"); TMP.mkdir(parents=True, exist_ok=True)

# 模型轮换池（遇到429/超时自动切换）
os.environ["RAGAS_MODEL_POOL"] = "gpt-4.1-nano,gpt-5-nano,gpt-4o-mini,gpt-4.1-mini,gpt-3.5-turbo"
MODEL_POOL = [m.strip() for m in os.getenv("RAGAS_MODEL_POOL").split(",") if m.strip()]
CHUNK_SIZE = 150  # 分块大小（可按需调小）

In [ ]:
# ========== 读取数据 ==========
DF = pd.read_parquet(INP).copy()
print(f"Loaded retrieval-scored: rows={len(DF)}, cols={len(DF.columns)}")

# —— 保障列表列为 list[str]
def _as_list(x):
    if isinstance(x, list): return [str(t) for t in x if str(t).strip()]
    if isinstance(x, tuple): return [str(t) for t in x if str(t).strip()]
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray): return [str(t) for t in x.tolist() if str(t).strip()]
    except Exception: pass
    if x is None: return []
    s = str(x).strip()
    if s.startswith("[") and s.endswith("]"):
        try:
            obj = json.loads(s)
            if isinstance(obj, list): return [str(t) for t in obj if str(t).strip()]
        except Exception:
            try:
                obj = ast.literal_eval(s)
                if isinstance(obj, (list, tuple)): return [str(t) for t in obj if str(t).strip()]
            except Exception: pass
    return []

DF["oracle_ctx"]    = DF.get("oracle_ctx", [[]]*len(DF)).apply(_as_list)
DF["retrieved_ctx"] = DF.get("retrieved_ctx", [[]]*len(DF)).apply(_as_list)

In [ ]:
# —— 如缺少 top-k（_ref_topk/_ctx_topk），即时构建（省 token）
import tiktoken
from rouge_score import rouge_scorer
ENC = tiktoken.get_encoding("cl100k_base")
def _tok_len(s): return len(ENC.encode(s or ""))
def _truncate(s, max_toks):
    ids = ENC.encode(s or "");
    return s if len(ids)<=max_toks else ENC.decode(ids[:max_toks])
_scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
def _select_topk(evs, q, a, k=3, budget=500, per_ctx=180):
    evs = [e for e in (evs or []) if isinstance(e,str) and e.strip()]
    if not evs: return []
    scored=[]
    for ev in evs:
        s = 0.6*_scorer.score(q,ev)["rougeL"].fmeasure + 0.4*_scorer.score(a,ev)["rougeL"].fmeasure
        scored.append((s,ev))
    scored.sort(key=lambda x:x[0], reverse=True)
    out, tot = [], 0
    for _,ev in scored[:k*3]:
        ev2=_truncate(ev, per_ctx); t=_tok_len(ev2)
        if tot+t <= budget: out.append(ev2); tot+=t
        if len(out)>=k: break
    if not out and evs: out=[_truncate(evs[0], min(300, budget))]
    return out[:k]

if "_ref_topk" not in DF.columns:  DF["_ref_topk"] = DF.apply(lambda r: _select_topk(r["oracle_ctx"],    r["question"], r["answer"]), axis=1)
if "_ctx_topk" not in DF.columns:  DF["_ctx_topk"] = DF.apply(lambda r: _select_topk(r["retrieved_ctx"], r["question"], r["answer"]), axis=1)


In [ ]:
# ========== RAGAS：Oracle only（answer_relevancy / faithfulness） ==========
from datasets import Dataset
from ragas import evaluate
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas.metrics import answer_relevancy as M_ANS_REL
from ragas.metrics import faithfulness    as M_FAITH

emb = HuggingFaceEmbeddings(model_name="intfloat/e5-base-v2")

def _make_llm(model_name):
    try:
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=model_name, temperature=0)
    except Exception:
        from langchain.chat_models import ChatOpenAI
        return ChatOpenAI(model_name=model_name, temperature=0)

def _eval_oracle_block(df_block, model_name: str) -> pd.DataFrame:
    data = {
        "user_input":         df_block["question"].astype(str).tolist(),
        "response":           df_block["answer"].astype(str).tolist(),
        "retrieved_contexts": df_block["_ref_topk"].tolist(),  # 用 oracle 证据评 faithful
    }
    ds = Dataset.from_dict(data)
    llm = _make_llm(model_name)
    res = evaluate(ds, metrics=[M_ANS_REL, M_FAITH], llm=llm, embeddings=emb)
    return res.to_pandas()

def _eval_with_pool_oracle(df_block) -> pd.DataFrame:
    last_err=None
    for m in MODEL_POOL:
        try:
            pdf = _eval_oracle_block(df_block, m)
            print(f"✅ oracle block with {m}")
            return pdf
        except Exception as e:
            last_err=e
            s=str(e).lower()
            if ("rate limit" in s) or ("429" in s) or ("timeout" in s):
                print(f"↪️ rate/timeout on {m}; trying next…")
                continue
            else:
                print(f"❗ non-retryable on {m}: {e}"); break
    raise RuntimeError(f"All models failed (oracle). Last error: {last_err}")

def run_ragas_oracle_only(df: pd.DataFrame, chunk_size=150) -> pd.DataFrame:
    out = df.copy()
    parts=[]
    for i in range(0, len(df), chunk_size):
        lo, hi = i, min(i+chunk_size, len(df))
        blk = df.iloc[lo:hi]
        cache = TMP / f"ragas_oracle_{lo}_{hi}.parquet"
        if cache.exists():
            try:
                pdf = pd.read_parquet(cache); pdf.index = blk.index
                print(f"⏭️  loaded oracle cache {lo}-{hi}")
                parts.append(pdf); continue
            except Exception: pass
        pdf = _eval_with_pool_oracle(blk)
        pdf.index = blk.index
        pdf.to_parquet(cache, index=False)
        parts.append(pdf)
    O = pd.concat(parts).sort_index()
    if "answer_relevancy" in O: out["oracle::answer_relevancy"] = O["answer_relevancy"].values
    if "faithfulness"     in O: out["oracle::faithfulness"]     = O["faithfulness"].values
    return out

print("▶️ Running RAGAS (oracle)…")
DF = run_ragas_oracle_only(DF, chunk_size=CHUNK_SIZE)
print("✔️ Oracle RAGAS done.")

In [ ]:
# ========== E5-base-v2 alignment：q→a, (q+oracle)→a, (q+retrieval)→a ==========
from sentence_transformers import SentenceTransformer
from numpy.linalg import norm

e5 = SentenceTransformer("intfloat/e5-base-v2", device="cuda" if torch.cuda.is_available() else "cpu")

def _cos(a, b):
    an = norm(a); bn = norm(b)
    if an==0 or bn==0: return float("nan")
    return float(np.dot(a, b) / (an*bn))

def _mk_query(text, ctx=None):
    if ctx and isinstance(ctx, (list,tuple)) and len(ctx)>0:
        joined = " ".join([c for c in ctx if isinstance(c,str) and c.strip()])
        return f"query: {text} [SEP] {joined}"
    return f"query: {text}"

def _mk_passage(ans): return f"passage: {ans}"

BATCH = 256
q_texts = DF["question"].astype(str).tolist()
a_texts = DF["answer"].astype(str).tolist()

q_e5, a_e5 = [], []
for s in range(0, len(DF), BATCH):
    q_e5 += list(e5.encode([_mk_query(t) for t in q_texts[s:s+BATCH]],
                           normalize_embeddings=True, show_progress_bar=False))
    a_e5 += list(e5.encode([_mk_passage(t) for t in a_texts[s:s+BATCH]],
                           normalize_embeddings=True, show_progress_bar=False))

DF["e5_align_q_to_a"] = [ _cos(q_e5[i], a_e5[i]) for i in range(len(DF)) ]

qo_e5 = []
for s in range(0, len(DF), BATCH):
    qo_e5 += list(e5.encode([_mk_query(q_texts[j], DF["_ref_topk"].iloc[s+j])
                              for j in range(min(BATCH, len(DF)-s))],
                             normalize_embeddings=True, show_progress_bar=False))
DF["e5_align_q_oracle_to_a"] = [ _cos(qo_e5[i], a_e5[i]) for i in range(len(DF)) ]

qr_e5 = []
for s in range(0, len(DF), BATCH):
    qr_e5 += list(e5.encode([_mk_query(q_texts[j], DF["_ctx_topk"].iloc[s+j])
                              for j in range(min(BATCH, len(DF)-s))],
                             normalize_embeddings=True, show_progress_bar=False))
DF["e5_align_q_retrieval_to_a"] = [ _cos(qr_e5[i], a_e5[i]) for i in range(len(DF)) ]

print("✔️ E5 alignment done.")

In [ ]:
# ========== AIS（RoBERTa MNLI）：句子级蕴含均值 → 1 - entail_mean ==========
from transformers import AutoTokenizer, AutoModelForSequenceClassification

NLI_NAME = "roberta-large-mnli"
tok = AutoTokenizer.from_pretrained(NLI_NAME)
mdl = AutoModelForSequenceClassification.from_pretrained(NLI_NAME)
_device = "cuda" if torch.cuda.is_available() else "cpu"
mdl.to(_device).eval()

def _sent_split(text, min_len=12):
    if not isinstance(text,str): return []
    parts = re.split(r'(?<=[\.\?\!。！？])\s+', text.strip())
    return [p.strip() for p in parts if len(p.strip())>=min_len]

@torch.inference_mode()
def _entail_prob(premise: str, hypothesis: str, max_length: int = 512) -> float:
    if not premise or not hypothesis: return 0.0
    enc = tok(premise, hypothesis, truncation=True, max_length=max_length,
              padding=False, return_tensors="pt").to(_device)
    logits = mdl(**enc).logits
    probs = torch.softmax(logits, dim=-1).squeeze(0)  # [contradiction, neutral, entailment]
    return float(probs[2].item())

def _ais_for_row(q, ctx_list, a):
    sents = _sent_split(a)
    if not sents: return float("nan")
    ctx = " ".join([c for c in (ctx_list or []) if isinstance(c,str) and c.strip()])
    premise = f"Question: {q}\nContext:\n{ctx}" if ctx else f"Question: {q}"
    ps = [ _entail_prob(premise, s) for s in sents ]
    if not ps: return float("nan")
    return 1.0 - float(np.mean(ps))  # 不一致度（越高越不一致）

DF["ais_oracle"] = [
    _ais_for_row(DF["question"].iloc[i], DF["_ref_topk"].iloc[i], DF["answer"].iloc[i])
    for i in tqdm(range(len(DF)), desc="AIS oracle")
]
DF["ais_retrieval"] = [
    _ais_for_row(DF["question"].iloc[i], DF["_ctx_topk"].iloc[i], DF["answer"].iloc[i])
    if isinstance(DF["_ctx_topk"].iloc[i], (list,tuple)) and len(DF["_ctx_topk"].iloc[i])>0 else np.nan
    for i in tqdm(range(len(DF)), desc="AIS retrieval")
]
DF["ais"] = DF["ais_retrieval"].where(~DF["ais_retrieval"].isna(), DF["ais_oracle"])
print("✔️ AIS done.")


In [ ]:
# ==== 保存“RAGAS（oracle）+ E5 对齐”结果 ====
import pandas as pd
from pathlib import Path

# 1) 读取当前 DF（若内存中没有 DF，则从 retrieval 结果载入）
try:
    _DF = DF.copy()
    print("✅ 使用内存中的 DF。")
except NameError:
    INP = Path("/Volumes/main/default/thesis_project/Evaluation/Eval_Test/qasper_all_models_unified.scored.retrieval.1.5.parquet")
    _DF = pd.read_parquet(INP)
    print(f"✅ 从磁盘载入：{INP}")

# 2) 检查我们要保存的列
ragas_oracle_cols = ["oracle::answer_relevancy", "oracle::faithfulness"]
e5_cols = ["e5_align_q_to_a", "e5_align_q_oracle_to_a", "e5_align_q_retrieval_to_a"]
keys = ["question_id", "model", "question", "answer"]

missing = [c for c in (ragas_oracle_cols + e5_cols) if c not in _DF.columns]
if missing:
    print("⚠️ 以下列当前不存在（可能尚未完成计算）：", missing)

# 3) 输出路径（一个“完整版”，一个“精简版”）
OUT_FULL = Path("/Volumes/main/default/thesis_project/Evaluation/Eval_Test/qasper_all_models_unified.scored.oracle_e5.1.5.parquet")
OUT_SLIM = Path("/Volumes/main/default/thesis_project/Evaluation/Eval_Test/qasper_all_models_unified.oracle_e5.slim.1.5.parquet")

# 4) 保存完整版（保留你现在 DF 的所有列，便于后续继续追加 AIS 或其它指标）
OUT_FULL.parent.mkdir(parents=True, exist_ok=True)
_df_full = _DF.copy()
_df_full.to_parquet(OUT_FULL, index=False)
print("💾 已保存完整版：", OUT_FULL)

# 5) 保存精简版（只包含主键 + oracle ragas + e5 指标）
present_cols = [c for c in (keys + ragas_oracle_cols + e5_cols) if c in _DF.columns]
_df_slim = _DF[present_cols].copy()
_df_slim.to_parquet(OUT_SLIM, index=False)
print("💾 已保存精简版：", OUT_SLIM)

# 6) 简要核验
print("\n— 非空计数（oracle + e5）：")
for c in ragas_oracle_cols + e5_cols:
    if c in _DF.columns:
        print(f"{c:>30s} : {_DF[c].notna().sum()} / {len(_DF)}")
    else:
        print(f"{c:>30s} : (缺列)")

# 可选：按模型看均值
if "model" in _DF.columns:
    cols_show = [c for c in (ragas_oracle_cols + e5_cols) if c in _DF.columns]
    print("\n— 按模型均值（部分列）：")
    display(_DF.groupby("model")[cols_show].mean(numeric_only=True).round(4))


In [ ]:
# ================== 快速 AIS（E5 语义覆盖版）==================
# 原理：把答案按句切分 -> 句向量；把来源段落向量化 -> 取每句对所有来源的最大相似度；
#      把相似度 rescale 到 [0,1] 后与阈值比较；最终取 “被覆盖的句子占比” 作为 AIS。
# 速度：相比 MNLI 版快一个量级以上；GPU 更快。

!pip -q install sentence-transformers --upgrade
import re, numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer, util

# 1) 准备 E5 模型（若你上文已有 e5，可直接用 e5；否则这里再加载一次）
try:
    _ = e5  # noqa
    st_model = e5  # 复用上文 e5-base-v2
except NameError:
    st_model = SentenceTransformer("intfloat/e5-base-v2", device="cuda" if torch.cuda.is_available() else "cpu")

# 2) 工具函数
def sent_split(text: str, min_len: int = 12):
    if not isinstance(text, str): return []
    parts = re.split(r'(?<=[\.\?\!。！？])\s+', text.strip())
    return [p.strip() for p in parts if len(p.strip()) >= min_len]

def _ensure_list_str(x):
    if isinstance(x, (list, tuple)):
        return [str(t) for t in x if isinstance(t, (str, bytes)) and str(t).strip()]
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray):
            return [str(t) for t in x.tolist() if str(t).strip()]
    except Exception:
        pass
    return []

def ais_score(answer: str, sources, sent_th: float = 0.70) -> float:
    sents = sent_split(answer or "")
    if not sents: return 0.0
    srcs = _ensure_list_str(sources)
    if not srcs: return 0.0
    # 向量化（一次性，自动批处理）
    ans_emb = st_model.encode(sents, normalize_embeddings=True, convert_to_tensor=True, show_progress_bar=False)
    src_emb = st_model.encode(srcs,  normalize_embeddings=True, convert_to_tensor=True, show_progress_bar=False)
    # 余弦相似度矩阵 [num_sents, num_srcs]
    sims = util.cos_sim(ans_emb, src_emb)  # [-1, 1]
    # 每句最大相似度 → [0,1] 归一后判断覆盖
    max_per_sent = sims.max(dim=1).values.detach().cpu().numpy()
    max_per_sent = (max_per_sent + 1.0) / 2.0
    return float((max_per_sent >= sent_th).mean())

# 3) 选择 AIS 来源（优先使用已经构建好的 top-k，避免把冗长原始上下文全拼进去）
def pick_sources(row: pd.Series, mode: str):
    if mode == "retrieved":
        src = row["_ctx_topk"] if "_ctx_topk" in row and _ensure_list_str(row["_ctx_topk"]) else row.get("retrieved_ctx", [])
    else:  # oracle
        src = row["_ref_topk"] if "_ref_topk" in row and _ensure_list_str(row["_ref_topk"]) else row.get("oracle_ctx", [])
    src = _ensure_list_str(src)
    # 保险：如果为空，尝试 citations（如果你有）
    if (not src) and ("citations" in row):
        src = _ensure_list_str(row["citations"])
    return src

# 4) 计算 AIS（oracle / retrieval / 综合）
tqdm.pandas(desc="AIS (E5) oracle")
DF["ais_oracle"] = DF.progress_apply(lambda r: ais_score(r["answer"], pick_sources(r, "oracle"), 0.70), axis=1)

tqdm.pandas(desc="AIS (E5) retrieval")
DF["ais_retrieval"] = DF.progress_apply(
    lambda r: ais_score(r["answer"], pick_sources(r, "retrieved"), 0.70) if len(pick_sources(r, "retrieved"))>0 else np.nan,
    axis=1
)

DF["ais"] = DF["ais_retrieval"].where(~DF["ais_retrieval"].isna(), DF["ais_oracle"])
print("✅ AIS (E5) done.")

In [ ]:
# ========== 保存 + 快检 ==========
OUT.parent.mkdir(parents=True, exist_ok=True)
DF.to_parquet(OUT, index=False)
print("💾 Saved:", OUT)

cols = [
  "oracle::answer_relevancy","oracle::faithfulness",
  "retrieval::answer_relevancy","retrieval::faithfulness",
  "retrieval::context_precision","retrieval::context_recall",
  "e5_align_q_to_a","e5_align_q_oracle_to_a","e5_align_q_retrieval_to_a",
  "ais_oracle","ais_retrieval","ais"
]
exist = [c for c in cols if c in DF.columns]
print("\nNon-null counts:")
print(DF[exist].notna().sum())
if "model" in DF.columns:
    print("\nPer-model means (head):")
    display(DF.groupby("model")[exist].mean(numeric_only=True).round(4).head())